**Important: read the following before you hit "Run all"**

Model checkpoints will be saved and loaded locally.

This notebook uses Google Colab. So, `local` means local disk used in the current Google Colab session.

To persist model checkpoints across sessions, this notebook saves the local copy to Google Drive after training.

To load a model back, this notebook copies model checkpoints from Google Drive back to Google Colab.

To transfer data between Google Colab and Google Drive, Google Drive is mounted under "/content/drive" and our data can be accessed via "/content/drive/MyDrive" in Google Colab.

**Checkpoints needed for training:**

R1 checkpoints on Google Drive: /checkpoints/r1_checkpoints/r1-sft-best

E1 checkpoints on Google Drive: /checkpoints/sft_sst2_gpt2_E1-1_2logits

**Checkpoints saved after training:**

R4 checkpoints on Google Drive: /checkpoints/r4_checkpoints

**Note**: Change these locations and files operations to match your environments as necessary. For example, you may mount your Google Drive, then load and save checkpoints directly from Google Drive. Though, load & save on local + copy before load & save after training between Google Colab and Google Drive as described above is preferred.

In [ ]:
# First, mount Drive (run in a cell)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copy R1 & E1 from Google Drive to local

!cp -r drive/MyDrive/checkpoints/r1_checkpoints/ .

!cp -r drive/MyDrive/checkpoints/sft_sst2_gpt2_E1-1_2logits .

In [ ]:
# Imports and configuration
import json
import math
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2Model,
    GPT2Config,
)

# --------------------------------------------------
# Paths
# --------------------------------------------------

R1_MODEL_PATH = "./r1_checkpoints/r1-sft-best"

E1_CHECKPOINT = "./sft_sst2_gpt2_E1-1_2logits"
E1_CHECKPOINT_WEIGHTS_FILE = "final_model_weights.pth"


OUTPUT_DIR = Path("./r4_checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_DIR = OUTPUT_DIR / "r4-ppo-final"

# --------------------------------------------------
# Training configuration
# --------------------------------------------------

# Make sure NUM_TRAIN_PROMPTS <= R3_TRAIN_PROMPTS (see a few cells below)
NUM_TRAIN_PROMPTS = 1000

BATCH_SIZE = 8
MINIBATCH_SIZE = 4

PPO_EPOCHS = 4
MAX_NEW_TOKENS = 40

LEARNING_RATE = 1e-6

BETA = 0.04
CLIP_RANGE = 0.2
VALUE_CLIP_RANGE = 0.2
VALUE_COEF = 0.1

GAMMA = 1.0
GAE_LAMBDA = 0.95

TEMPERATURE = 0.9
TOP_P = 1.0

MAX_E1_LENGTH = 128

CHECKPOINT_EVERY = 25

SEED = 42

# --------------------------------------------------
# Device and reproducibility
# --------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

Device: cuda


In [ ]:
# Load the tokenizer and R1 policy

tokenizer = AutoTokenizer.from_pretrained(R1_MODEL_PATH)

# GPT-2 has no dedicated padding token.
tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

policy = GPT2LMHeadModel.from_pretrained(
    R1_MODEL_PATH
).to(DEVICE)

reference_policy = GPT2LMHeadModel.from_pretrained(
    R1_MODEL_PATH
).to(DEVICE)

reference_policy.eval()

for param in reference_policy.parameters():
    param.requires_grad = False

# Disable dropout to keep rollout and optimization
# probability calculations consistent.
policy.eval()
reference_policy.eval()

print("Policy parameters:",
      sum(p.numel() for p in policy.parameters()))

print("Reference policy parameters:",
      sum(p.numel() for p in reference_policy.parameters()))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Policy parameters: 124439808
Reference policy parameters: 124439808


In [ ]:
# Add the value head

class ValueHead(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        self.value = nn.Linear(hidden_size, 1)

        nn.init.normal_(
            self.value.weight,
            mean=0.0,
            std=0.02
        )

        nn.init.zeros_(self.value.bias)

    def forward(self, hidden_states):
        return self.value(hidden_states).squeeze(-1)


value_head = ValueHead(
    policy.config.n_embd
).to(DEVICE)

print(value_head)

ValueHead(
  (value): Linear(in_features=768, out_features=1, bias=True)
)


Our trainable parameters now consist of:

- GPT-2's language-model parameters.
- The additional value-head parameters.

The reference policy and E1 classifier remain frozen.

In [ ]:
# Load E1 model

MODEL_NAME = "openai-community/gpt2"

#E1_CHECKPOINT = "./sft_sst2_gpt2_E1-1_2logits"
#E1_CHECKPOINT_WEIGHTS_FILE = "final_model_weights.pth"

# Load E1 - the sentiment classifier, used to classify sentiment of texts to determine "chosen" and "rejected"
# can be thinked of as a reward model


# Load the tokenizer (it knows how to reconstruct itself from the folder)
# We may not need this, double check ...
e1_tokenizer = AutoTokenizer.from_pretrained(E1_CHECKPOINT)


# ============================================================
# Model
# ============================================================

class GPT2SentimentClassifier(nn.Module):

    def __init__(self, model_name, num_labels=2):
        super().__init__()

        # GPT-2 WITHOUT the language-model head.
        self.gpt2 = GPT2Model.from_pretrained(model_name)

        hidden_size = self.gpt2.config.hidden_size

        # Binary classification head:
        #
        # [B, 768] -> [B, 2]
        self.classifier = nn.Linear(
            hidden_size,
            num_labels,
        )


    def forward(self, input_ids, attention_mask, labels=None):

        # ----------------------------------------------------
        # GPT-2
        #
        # last_hidden_state:
        # [B, T, 768]
        # ----------------------------------------------------

        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state

        # ----------------------------------------------------
        # Find the position of the last real token
        #
        # attention_mask:
        #
        #   1 1 1 1 1 0 0
        #
        # last real token = position 4
        # ----------------------------------------------------

        sequence_lengths = attention_mask.sum(dim=1) - 1

        batch_indices = torch.arange(
            hidden_states.size(0),
            device=hidden_states.device,
        )

        # [B, 768]
        pooled_hidden_states = hidden_states[
            batch_indices,
            sequence_lengths,
        ]

        # ----------------------------------------------------
        # Classification head
        #
        # [B, 768] -> [B, 2]
        # ----------------------------------------------------

        logits = self.classifier(
            pooled_hidden_states
        )


        # ----------------------------------------------------
        # Classification loss
        # ----------------------------------------------------

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits,
        }

# 2. Initialize our model class (we know the architecture)
e1_model = GPT2SentimentClassifier(
    MODEL_NAME,
    num_labels=2,
)

# 3. Load the saved weights and freeze
e1_checkpoint = torch.load(os.path.join(E1_CHECKPOINT, E1_CHECKPOINT_WEIGHTS_FILE), weights_only=True)
e1_model.load_state_dict(e1_checkpoint['model_state_dict'])
e1_model.to(DEVICE)
e1_model.eval()

for parameter in e1_model.parameters():
    parameter.requires_grad = False

print("E1 model (sentiment classifier) loaded.")
print("Device:", DEVICE)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

E1 model (sentiment classifier) loaded.
Device: cuda


In [ ]:
# Reuse the R3 prompt datasett

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2Model,
    set_seed,
)

# --------------------------------------------------
# Configuration
# --------------------------------------------------

R3_TRAIN_PROMPTS = 1000
R3_VALIDATION_PROMPTS = 100

SEED = 42

SENTIMENT_NAMES = {
    0: "negative",
    1: "positive"
}

set_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device: {DEVICE}")

# --------------------------------------------------
# Create sentiment-conditioned prompts
# --------------------------------------------------

def create_grpo_prompts(split, num_examples, seed):

    source_dataset = load_dataset(
        "stanfordnlp/sst2",
        split=split
    )

    rng = random.Random(seed)

    prompt_records = []

    for example in source_dataset:

        words = example["sentence"].split()

        if len(words) < 8:
            continue

        # Randomly retain 50%-70% as the prefix.
        min_split = max(4, math.ceil(len(words) * 0.50))
        max_split = min(
            len(words) - 3,
            math.floor(len(words) * 0.70)
        )

        if min_split > max_split:
            continue

        split_index = rng.randint(min_split, max_split)

        prefix = " ".join(words[:split_index])

        sentiment = SENTIMENT_NAMES[example["label"]]

        prompt = (
            f"Sentiment: {sentiment}\n"
            f"Continue the following review:\n"
            f"{prefix}"
        )

        prompt_records.append({
            "prompt": prompt
        })

        if len(prompt_records) >= num_examples:
            break

    return Dataset.from_list(prompt_records)


train_dataset = create_grpo_prompts(
    split="train",
    num_examples=R3_TRAIN_PROMPTS,
    seed=SEED
)

validation_dataset = create_grpo_prompts(
    split="validation",
    num_examples=R3_VALIDATION_PROMPTS,
    seed=SEED + 1
)

print("Training prompts:", len(train_dataset))
print("Validation prompts:", len(validation_dataset))

print("\nExample prompt:")
print(train_dataset[0]["prompt"])

Device: cuda
Training prompts: 1000
Validation prompts: 100

Example prompt:
Sentiment: positive
Continue the following review:
that loves its characters and communicates something rather


In [ ]:
# The PPO reward function needs three pieces of information:
# - The full prompt.
# - The requested sentiment.
# - The original review prefix.

# We want to retain the same training and validation examples as R3.
# Since our R3 dataset contains only a prompt column, we need to recover the other two fields from the prompt text.

def parse_prompt(prompt):

    lines = prompt.splitlines()

    sentiment = lines[0].split(":", 1)[1].strip()

    prefix = prompt.split(
        "Continue the following review:\n",
        1
    )[1].strip()

    return {
        "prompt": prompt,
        "sentiment": sentiment,
        "prefix": prefix
    }


# Use the existing R3 datasets.
# train_dataset and validation_dataset should already
# contain the R3 prompt examples.

train_records = [
    parse_prompt(x["prompt"])
    for x in train_dataset
]

validation_records = [
    parse_prompt(x["prompt"])
    for x in validation_dataset
]

train_records = train_records[:NUM_TRAIN_PROMPTS]

print("Training prompts:", len(train_records))
print("Validation prompts:", len(validation_records))

print(train_records[0])

Training prompts: 1000
Validation prompts: 100
{'prompt': 'Sentiment: positive\nContinue the following review:\nthat loves its characters and communicates something rather', 'sentiment': 'positive', 'prefix': 'that loves its characters and communicates something rather'}


In [ ]:
# Implement the reward function

def calculate_reward(
    tokenizer,
    sentiment,
    prefix,
    completion
):

    full_review = (
        prefix + completion
    ).strip()

    inputs = tokenizer(
        full_review,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_E1_LENGTH
    ).to(DEVICE)

    with torch.no_grad():

        outputs = e1_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        probabilities = torch.softmax(
            outputs["logits"],
            dim=-1
        )

    target_label = (
        1 if sentiment == "positive" else 0
    )

    return probabilities[0, target_label].item()

The reward function follows exactly the same definition as R3:

$$ r(x,y)=P_{\text{E1}}(\text{requested sentiment}\mid x+y) $$

Notice that we feed E1 the original review prefix plus the generated continuation, not the instruction text.

In [ ]:
# Sanity-check the reward function with an example

examples = [
    (
        "positive",
        "that loves its characters and communicates something",
        " very special about humankind"
    ),
    (
        "negative",
        "that loves its characters and communicates something",
        " that is really, truly unpleasant"
    )
]

for sentiment, prefix, completion in examples:

    reward = calculate_reward(
        e1_tokenizer,
        sentiment,
        prefix,
        completion
    )

    print("Sentiment:", sentiment)
    print("Completion:", completion)
    print("Reward:", round(reward, 4))
    print()

Sentiment: positive
Completion:  very special about humankind
Reward: 0.6166

Sentiment: negative
Completion:  that is really, truly unpleasant
Reward: 0.5819



**Implement PPO rollouts and advantage estimation**

This is the heart of R4.

For each prompt, PPO will:

1. Generate a completion using the current policy.
2. Calculate its E1 reward.
3. Calculate policy and reference log-probabilities for every generated token.
4. Calculate per-token KL penalties.
5. Use the value head and GAE to estimate advantages.
6. Perform several PPO updates on the collected rollout batch.


In [ ]:
# Forward pass and rollout generation

def get_logprobs_and_values(
    model,
    value_head,
    input_ids,
    prompt_length,
    response_length
):
    """
    Calculate log probabilities and value predictions
    for the generated response tokens.

    Logits at position t predict token t+1.
    """

    attention_mask = torch.ones_like(input_ids)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True
    )

    logits = outputs.logits[:, :-1, :]
    hidden = outputs.hidden_states[-1][:, :-1, :]

    start = prompt_length - 1
    end = start + response_length

    response_logits = logits[:, start:end, :]
    response_hidden = hidden[:, start:end, :]

    response_ids = input_ids[
        :,
        prompt_length:prompt_length + response_length
    ]

    response_logits = response_logits / TEMPERATURE

    log_probs = F.log_softmax(
        response_logits,
        dim=-1
    )

    selected_log_probs = log_probs.gather(
        dim=-1,
        index=response_ids.unsqueeze(-1)
    ).squeeze(-1)

    values = value_head(
        response_hidden
    )

    return selected_log_probs.squeeze(0), values.squeeze(0)


def generate_rollout(record):

    prompt = record["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(DEVICE)

    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        generated = policy.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response_ids = generated[
        0,
        prompt_length:
    ]

    # Keep the first EOS token, if generated.
    eos_positions = (
        response_ids == tokenizer.eos_token_id
    ).nonzero(as_tuple=True)[0]

    if len(eos_positions) > 0:

        response_ids = response_ids[
            :eos_positions[0].item() + 1
        ]

    if response_ids.numel() == 0:
        return None

    response_length = response_ids.shape[0]

    full_ids = torch.cat(
        [
            inputs["input_ids"][0],
            response_ids
        ]
    ).unsqueeze(0)

    completion = tokenizer.decode(
        response_ids,
        skip_special_tokens=True
    )

    # Reward from frozen E1.
    reward_score = calculate_reward(
        e1_tokenizer,
        record["sentiment"],
        record["prefix"],
        completion
    )

    # Old policy log probabilities and values.
    with torch.no_grad():

        old_logprobs, old_values = (
            get_logprobs_and_values(
                policy,
                value_head,
                full_ids,
                prompt_length,
                response_length
            )
        )

        ref_logprobs, _ = (
            get_logprobs_and_values(
                reference_policy,
                value_head,
                full_ids,
                prompt_length,
                response_length
            )
        )

    old_logprobs = old_logprobs.detach()
    old_values = old_values.detach()
    ref_logprobs = ref_logprobs.detach()

    # Per-token KL penalty.
    kl = old_logprobs - ref_logprobs

    token_rewards = -BETA * kl

    # Add the scalar E1 reward at the final response token.
    token_rewards[-1] += reward_score

    return {
        "input_ids": full_ids[0].detach(),
        "prompt_length": prompt_length,
        "response_length": response_length,
        "completion": completion,
        "reward": reward_score,
        "kl": kl.detach(),
        "old_logprobs": old_logprobs,
        "old_values": old_values,
        "token_rewards": token_rewards.detach()
    }

We calculate GAE backwards through the generated response.

For each token:
$$
\delta_t=r_t+\gamma V(s_{t+1})-V(s_t)
$$

$$
A_t=\delta_t+\gamma\lambda A_{t+1}
$$

The return target is:
$$
R_t=A_t+V(s_t)
$$

In [ ]:
# Generalized Advantage Estimation

def calculate_gae(rollout):

    rewards = rollout["token_rewards"]
    values = rollout["old_values"]

    advantages = torch.zeros_like(rewards)

    gae = 0.0

    for t in reversed(range(len(rewards))):

        if t == len(rewards) - 1:
            next_value = 0.0
        else:
            next_value = values[t + 1]

        delta = (
            rewards[t]
            + GAMMA * next_value
            - values[t]
        )

        gae = (
            delta
            + GAMMA * GAE_LAMBDA * gae
        )

        advantages[t] = gae

    returns = advantages + values

    rollout["advantages"] = advantages.detach()
    rollout["returns"] = returns.detach()

    return rollout

A key detail: our task uses a sequence-level reward, so the E1 reward is added to the last generated token. The KL penalty is distributed across the response tokens.

This is an outcome-reward PPO setup, rather than a token-by-token supervised reward setup.

PPO Optimization

In [ ]:
# Optimizer

optimizer = torch.optim.AdamW(
    list(policy.parameters())
    + list(value_head.parameters()),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

**Batch preparation**

We need to support variable prompt and completion lengths. The following function right-pads sequences and aligns the old log-probabilities, value predictions, advantages, and returns with their corresponding response tokens.


In [ ]:
# Batch preparation

def prepare_ppo_batch(samples):

    batch_size = len(samples)

    max_length = max(
        len(s["input_ids"])
        for s in samples
    )

    input_ids = torch.full(
        (batch_size, max_length),
        tokenizer.pad_token_id,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_length),
        dtype=torch.long
    )

    shifted_length = max_length - 1

    response_mask = torch.zeros(
        (batch_size, shifted_length),
        dtype=torch.bool
    )

    old_logprobs = torch.zeros(
        (batch_size, shifted_length)
    )

    old_values = torch.zeros(
        (batch_size, shifted_length)
    )

    advantages = torch.zeros(
        (batch_size, shifted_length)
    )

    returns = torch.zeros(
        (batch_size, shifted_length)
    )

    for i, sample in enumerate(samples):

        ids = sample["input_ids"]

        seq_length = len(ids)
        prompt_length = sample["prompt_length"]
        response_length = sample["response_length"]

        input_ids[i, :seq_length] = ids
        attention_mask[i, :seq_length] = 1

        # Logit/value position t predicts response token t+1.
        start = prompt_length - 1
        end = start + response_length

        response_mask[i, start:end] = True

        old_logprobs[i, start:end] = (
            sample["old_logprobs"].cpu()
        )

        old_values[i, start:end] = (
            sample["old_values"].cpu()
        )

        advantages[i, start:end] = (
            sample["advantages"].cpu()
        )

        returns[i, start:end] = (
            sample["returns"].cpu()
        )

    return {
        "input_ids": input_ids.to(DEVICE),
        "attention_mask": attention_mask.to(DEVICE),
        "response_mask": response_mask.to(DEVICE),
        "old_logprobs": old_logprobs.to(DEVICE),
        "old_values": old_values.to(DEVICE),
        "advantages": advantages.to(DEVICE),
        "returns": returns.to(DEVICE)
    }

**PPO update**

This implements the clipped PPO policy objective and clipped value objective.

In [ ]:
# PPO update

def ppo_update(samples):

    batch = prepare_ppo_batch(samples)

    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]
    response_mask = batch["response_mask"]

    old_logprobs = batch["old_logprobs"]
    old_values = batch["old_values"]

    advantages = batch["advantages"]
    returns = batch["returns"]

    # Normalize advantages over valid response tokens.
    valid_advantages = advantages[response_mask]

    adv_mean = valid_advantages.mean()
    adv_std = valid_advantages.std(
        unbiased=False
    ).clamp_min(1e-8)

    advantages = (
        advantages - adv_mean
    ) / adv_std

    batch_size = input_ids.size(0)

    metrics = []

    for ppo_epoch in range(PPO_EPOCHS):

        indices = torch.randperm(batch_size)

        for start in range(
            0,
            batch_size,
            MINIBATCH_SIZE
        ):

            mb_indices = indices[
                start:start + MINIBATCH_SIZE
            ]

            mb_input_ids = input_ids[mb_indices]
            mb_attention_mask = attention_mask[mb_indices]

            mb_mask = response_mask[mb_indices]

            mb_old_logprobs = old_logprobs[mb_indices]
            mb_old_values = old_values[mb_indices]

            mb_advantages = advantages[mb_indices]
            mb_returns = returns[mb_indices]

            outputs = policy(
                input_ids=mb_input_ids,
                attention_mask=mb_attention_mask,
                output_hidden_states=True,
                use_cache=False,
                return_dict=True
            )

            logits = outputs.logits[:, :-1, :]

            hidden = outputs.hidden_states[-1][:, :-1, :]

            values = value_head(hidden)

            targets = mb_input_ids[:, 1:]

            logits = logits / TEMPERATURE

            log_probs = F.log_softmax(
                logits,
                dim=-1
            )

            new_logprobs = log_probs.gather(
                dim=-1,
                index=targets.unsqueeze(-1)
            ).squeeze(-1)

            # PPO probability ratio.
            ratio = torch.exp(
                new_logprobs - mb_old_logprobs
            )

            unclipped_objective = (
                ratio * mb_advantages
            )

            clipped_objective = (
                torch.clamp(
                    ratio,
                    1.0 - CLIP_RANGE,
                    1.0 + CLIP_RANGE
                )
                * mb_advantages
            )

            policy_loss = -torch.minimum(
                unclipped_objective,
                clipped_objective
            )[mb_mask].mean()

            # Clipped value loss.
            clipped_values = (
                mb_old_values
                + torch.clamp(
                    values - mb_old_values,
                    -VALUE_CLIP_RANGE,
                    VALUE_CLIP_RANGE
                )
            )

            value_loss_1 = (
                values - mb_returns
            ) ** 2

            value_loss_2 = (
                clipped_values - mb_returns
            ) ** 2

            value_loss = 0.5 * torch.maximum(
                value_loss_1,
                value_loss_2
            )[mb_mask].mean()

            loss = (
                policy_loss
                + VALUE_COEF * value_loss
            )

            optimizer.zero_grad()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                list(policy.parameters())
                + list(value_head.parameters()),
                max_norm=1.0
            )

            optimizer.step()

            with torch.no_grad():

                approx_kl = (
                    mb_old_logprobs - new_logprobs
                )[mb_mask].mean()

                clip_fraction = (
                    (ratio - 1.0).abs() > CLIP_RANGE
                )[mb_mask].float().mean()

            metrics.append({
                "policy_loss": policy_loss.item(),
                "value_loss": value_loss.item(),
                "approx_kl": approx_kl.item(),
                "clip_fraction": clip_fraction.item()
            })

    return {
        key: float(np.mean([
            m[key] for m in metrics
        ]))
        for key in metrics[0]
    }

A note on the implementation: the old_logprobs and old_values are collected before optimization and held fixed during all four PPO epochs. This is essential: PPO compares the updated policy against the policy that generated the rollout.

**Training loop, checkpointing, and validation**

**Save checkpoints**

We save both the language model and value head. The value head is necessary if we want to resume PPO training, although it is not needed for ordinary text generation after training.

In [ ]:
# Save checkpoints

def save_r4_checkpoint(path):

    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)

    policy.save_pretrained(
        path / "policy"
    )

    tokenizer.save_pretrained(
        path / "policy"
    )

    torch.save(
        value_head.state_dict(),
        path / "value_head.pt"
    )

    torch.save(
        optimizer.state_dict(),
        path / "optimizer.pt"
    )

    print("Saved checkpoint:", path)

**Validation reward**

We evaluate the current policy on the same 100 validation prompts used by R3.

In [ ]:
# Validation reward

def evaluate_policy(records):

    policy.eval()

    rewards = []
    completions = []

    for record in tqdm(
        records,
        desc="R4 validation"
    ):

        inputs = tokenizer(
            record["prompt"],
            return_tensors="pt",
            add_special_tokens=False
        ).to(DEVICE)

        prompt_length = inputs["input_ids"].shape[1]

        with torch.no_grad():

            generated = policy.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        response_ids = generated[
            0,
            prompt_length:
        ]

        eos_positions = (
            response_ids == tokenizer.eos_token_id
        ).nonzero(as_tuple=True)[0]

        if len(eos_positions) > 0:

            response_ids = response_ids[
                :eos_positions[0].item() + 1
            ]

        completion = tokenizer.decode(
            response_ids,
            skip_special_tokens=True
        )

        reward = calculate_reward(
            e1_tokenizer,
            record["sentiment"],
            record["prefix"],
            completion
        )

        rewards.append(reward)

        completions.append({
            "prompt": record["prompt"],
            "completion": completion,
            "reward": reward
        })

    return {
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "completions": completions
    }

**Main training loop**

In [ ]:
# Main training loop

metrics_path = OUTPUT_DIR / "r4_training_metrics.jsonl"

global_step = 0

training_records = train_records.copy()

random.shuffle(training_records)

num_batches = math.ceil(
    len(training_records) / BATCH_SIZE
)

progress = tqdm(
    range(num_batches),
    desc="R4 PPO training"
)

for batch_index in progress:

    start = batch_index * BATCH_SIZE

    batch_records = training_records[
        start:start + BATCH_SIZE
    ]

    # ------------------------------------------
    # 1. Collect online rollouts
    # ------------------------------------------

    rollouts = []

    for record in batch_records:

        rollout = generate_rollout(record)

        if rollout is not None:

            rollout = calculate_gae(rollout)

            rollouts.append(rollout)

    if len(rollouts) < 2:
        continue

    # ------------------------------------------
    # 2. PPO optimization
    # ------------------------------------------

    update_metrics = ppo_update(rollouts)

    global_step += 1

    mean_reward = float(np.mean([
        r["reward"] for r in rollouts
    ]))

    mean_kl = float(np.mean([
        r["kl"].mean().item() for r in rollouts
    ]))

    mean_length = float(np.mean([
        r["response_length"] for r in rollouts
    ]))

    log_entry = {
        "step": global_step,
        "reward": mean_reward,
        "kl": mean_kl,
        "mean_completion_length": mean_length,
        **update_metrics
    }

    with open(metrics_path, "a") as f:

        f.write(
            json.dumps(log_entry) + "\n"
        )

    progress.set_postfix({
        "reward": f"{mean_reward:.4f}",
        "kl": f"{mean_kl:.4f}",
        "policy_loss": f"{update_metrics['policy_loss']:.4f}",
        "value_loss": f"{update_metrics['value_loss']:.4f}"
    })

    # ------------------------------------------
    # 3. Periodic checkpoint
    # ------------------------------------------

    if global_step % CHECKPOINT_EVERY == 0:

        save_r4_checkpoint(
            OUTPUT_DIR / f"checkpoint-{global_step}"
        )

# ----------------------------------------------
# Final checkpoint
# ----------------------------------------------

save_r4_checkpoint(FINAL_MODEL_DIR)

print("R4 PPO training completed.")

R4 PPO training:   0%|          | 0/125 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint: r4_checkpoints/checkpoint-25


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint: r4_checkpoints/checkpoint-50


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint: r4_checkpoints/checkpoint-125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint: r4_checkpoints/r4-ppo-final
R4 PPO training completed.


One important practical note: this is a full-fine-tuning implementation. GPT-2 124M should be manageable on a Colab T4, but if you encounter CUDA out-of-memory errors, reduce BATCH_SIZE to 4 and MINIBATCH_SIZE to 2. Keep the rollout batch divisible by the minibatch size.

In [ ]:
# Final validation and results saving

validation_results = evaluate_policy(
    validation_records
)

print("\nR4 PPO validation results")

print(
    "Mean reward:",
    round(validation_results["mean_reward"], 4)
)

print(
    "Reward std:",
    round(validation_results["std_reward"], 4)
)

with open(
    OUTPUT_DIR / "r4_validation_results.json",
    "w"
) as f:

    json.dump(
        validation_results,
        f,
        indent=2
    )

print("\nSample generations:")

for item in validation_results["completions"][:10]:

    print("\nPrompt:", item["prompt"])
    print("Completion:", item["completion"])
    print("Reward:", round(item["reward"], 4))

R4 validation:   0%|          | 0/100 [00:00<?, ?it/s]


R4 PPO validation results
Mean reward: 0.6274
Reward std: 0.1703

Sample generations:

Prompt: Sentiment: positive
Continue the following review:
it 's a charming and
Completion:  delightful documentary
Reward: 0.6988

Prompt: Sentiment: positive
Continue the following review:
allows us to hope that nolan is poised to embark a major
Completion:  journey ,
Reward: 0.8216

Prompt: Sentiment: positive
Continue the following review:
the acting , costumes , music , cinematography and sound are
Completion:  wonderful .
Reward: 0.8231

Prompt: Sentiment: negative
Continue the following review:
it 's slow -- very ,
Completion:  very slow
Reward: 0.4594

Prompt: Sentiment: positive
Continue the following review:
although laced with humor and a few fanciful touches , the film is
Completion:  a complete masterpiece
Reward: 0.7667

Prompt: Sentiment: negative
Continue the following review:
or doing last year 's taxes with
Completion:  the smugest of all jokes
Reward: 0.7896

Prompt: Sentiment: po

Optional: One small generation setting to add, in both policy.generate() calls above, add:

```python
top_k=0,
```

This disables the default top-k sampling restriction, so rollout sampling is consistent with the temperature-scaled log-probabilities used in our PPO objective.

In [ ]:
# Loading the trained R4 model
# For normal generation, we only need the trained GPT-2 language model. The value head is not needed for inference.

R4_MODEL_PATH = (
    "./r4_checkpoints/r4-ppo-final/policy"
)

r4_tokenizer = AutoTokenizer.from_pretrained(
    R4_MODEL_PATH
)

r4_model = GPT2LMHeadModel.from_pretrained(
    R4_MODEL_PATH
).to(DEVICE)

r4_model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
# Test

# A common continuation generation function to be used for all experiments (models), making a comparison across them fairer

def generate_continuation(
    model,
    tokenizer,
    sentiment,
    prefix,
    max_new_tokens=40
):
    """
    Generate a sentiment-conditioned continuation.

    Args:
        model: Hugging Face causal language model.
        tokenizer: Corresponding tokenizer.
        sentiment: "positive" or "negative".
        prefix: Review prefix to continue.
        max_new_tokens: Maximum number of new tokens.

    Returns:
        Generated continuation (without the original prompt).
    """

    # Construct prompt exactly as in R1 and R2.
    prompt = (
        f"Sentiment: {sentiment}\n"
        f"Continue the following review:\n"
        f"{prefix}"
    )

    # Determine model device.
    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    model.eval()

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Remove prompt tokens to obtain continuation only.
    prompt_length = inputs["input_ids"].shape[1]

    continuation_ids = output_ids[0, prompt_length:]

    continuation = tokenizer.decode(
        continuation_ids,
        skip_special_tokens=True
    )

    return continuation

# Use the same prefix as we do for SFT, DPO, and GRPO

prefix = "that loves its characters and communicates something"

for sentiment in ["positive", "negative"]:

    continuation = generate_continuation(
        model=r4_model,
        tokenizer=r4_tokenizer,
        sentiment=sentiment,
        prefix=prefix,
        max_new_tokens=40
    )

    print(f"\nR4 - {sentiment.capitalize()}:")
    print(continuation)


R4 - Positive:
 beautiful and genuine

R4 - Negative:
 fundamentally creepy and frightening


In [ ]:
# Save r4 to Google Drive
# If we have enough storage to use on our Google Drive, save checkpoints and final model
!cp -r ./r4_checkpoints drive/MyDrive/checkpoints

# Otherwise, save final model & metrics
!cp -r ./r4_checkpoints/r4-ppo-final/ drive/MyDrive/checkpoints/r4_checkpoints

!cp -r ./r4_checkpoints/r4_training_metrics.jsonl drive/MyDrive/checkpoints/r4_checkpoints



**What to look for in R4 training results**

The following metrics are logged and saved to `r4_training_metrics.jsonl`:

```
Metric	            Purpose
Mean reward	        Ideally increases over training
Mean KL	            Remains reasonably controlled
Policy loss	        Stable optimization, not necessarily monotonically decreasing
Value loss	        Generally learns to predict returns
Approximate KL	    No excessively large policy updates
Clip fraction	      Some clipping can occur; persistent excessive clipping is undesirable
Completion length	  No pathological shortening or lengthening
```

A few cautions:

1. Value loss is not a measure of generation quality. It measures how well the value head estimates returns.
2. Reward improvement alone does not prove better text. E1 can be exploited just as it could in R3.
3. A low KL is not automatically good. If the policy barely moves from R1, it may not learn much.
4. PPO needs multiple optimization passes over each rollout. Our four PPO epochs provide that.

The current TRL PPO documentation also identifies reward, KL, policy loss, value loss, and clipping statistics as useful training diagnostics.